# Hyperparameter Search (Optuna + Hydra Sweeper)

This notebook runs Optuna-driven Hydra multirun searches for the MLP and CNN agents on `samegame_5x5c3s2`.

## Objective
- Optimize `training.optimize_metric=avg_return`
- Persist studies in SQLite
- Visualize studies with `optuna-dashboard`

## 0. Imports and Paths

In [2]:
import json
import os
import shlex
import signal
import subprocess
import sys
from pathlib import Path

import optuna

WORKDIR = Path.cwd()
DB_DIR = WORKDIR / "optuna"
DB_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = DB_DIR / "hypersearch.db"
DB_URL = f"sqlite:///{DB_PATH}"
dashboard_proc = None

print(f"Working directory: {WORKDIR}")
print(f"Optuna DB URL:    {DB_URL}")

Working directory: /home/f133r/projects/dyad_rl
Optuna DB URL:    sqlite:////home/f133r/projects/dyad_rl/optuna/hypersearch.db


## 1. Environment and Dependency Check

In [3]:
def _version(pkg: str) -> str:
    try:
        mod = __import__(pkg)
        return getattr(mod, '__version__', 'unknown')
    except Exception as exc:
        return f"missing ({exc})"

print(f"Python: {sys.version.split()[0]}")
print(f"hydra: {_version('hydra')}")
print(f"optuna: {_version('optuna')}")

cmd = [sys.executable, '-m', 'optuna_dashboard', '--help']
res = subprocess.run(cmd, capture_output=True, text=True)
print('optuna-dashboard CLI available:', res.returncode == 0)
if res.returncode != 0:
    print(res.stderr)

required = [
    WORKDIR / 'config' / 'sweep_mlp.yaml',
    WORKDIR / 'config' / 'sweep_cnn.yaml',
    WORKDIR / 'experiment.py',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")
print('Config files found.')

Python: 3.11.13
hydra: 1.3.2
optuna: 4.7.0
optuna-dashboard CLI available: False
/home/f133r/projects/dyad_rl/.venv/bin/python: No module named optuna_dashboard.__main__; 'optuna_dashboard' is a package and cannot be directly executed

Config files found.


## 2. Subprocess Helpers

In [4]:
def run_cmd(cmd: list[str], env_updates: dict[str, str] | None = None) -> int:
    env = os.environ.copy()
    if env_updates:
        env.update(env_updates)

    print('Running command:')
    print(' '.join(shlex.quote(c) for c in cmd))

    process = subprocess.Popen(
        cmd,
        cwd=str(WORKDIR),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')

    process.wait()
    print(f"Exit code: {process.returncode}")
    return int(process.returncode)

def build_sweep_cmd(config_name: str, extra_overrides: list[str] | None = None) -> list[str]:
    extra_overrides = extra_overrides or []
    return [
        sys.executable,
        'experiment.py',
        '--multirun',
        f'--config-name={config_name}',
        f'hydra.sweeper.storage={DB_URL}',
    ] + extra_overrides

## 3. MLP Search Cell (samegame_5x5c3s2)

This cell launches a dedicated Optuna sweep for the MLP agent.

- Default trials are pulled from `config/sweep_mlp.yaml` (80)
- Set `mlp_trials_override` for quick smoke runs

In [ ]:
mlp_trials_override: int | None = None # e.g., 3 for smoke, None for full config
mlp_extra = []
if mlp_trials_override is not None:
    mlp_extra.append(f'hydra.sweeper.n_trials={mlp_trials_override}')

# Optional speed/cost controls for sweeps:
# mlp_extra += ['training.total_episodes=15000', 'training.eval_interval=500', 'training.eval_episodes=50']

mlp_code = run_cmd(build_sweep_cmd('sweep_mlp', mlp_extra))
if mlp_code != 0:
    raise RuntimeError('MLP sweep failed. Inspect logs above.')

Running command:
/home/f133r/projects/dyad_rl/.venv/bin/python experiment.py --multirun --config-name=sweep_mlp hydra.sweeper.storage=sqlite:////home/f133r/projects/dyad_rl/optuna/hypersearch.db hydra.sweeper.n_trials=3
(null): No such file or directory
(null): No such file or directory
/home/f133r/projects/dyad_rl/.venv/lib/python3.11/site-packages/hydra_plugins/hydra_optuna_sweeper/_impl.py:56: FutureWarning: LogUniformDistribution has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :class:`~optuna.distributions.FloatDistribution` instead.
  return LogUniformDistribution(param.low, param.high)
/home/f133r/projects/dyad_rl/.venv/lib/python3.11/site-packages/hydra_plugins/hydra_optuna_sweeper/_impl.py:59: FutureWarning: UniformDistribution has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use :class:`~optuna.distributions.FloatD

## 4. CNN Search Cell (samegame_5x5c3s2)

This cell launches a dedicated Optuna sweep for the CNN agent. Keep `n_jobs=1` for GPU memory safety.

In [ ]:
cnn_trials_override: int | None = None  # e.g., 3 for smoke, None for full config
cnn_extra = []
if cnn_trials_override is not None:
    cnn_extra.append(f'hydra.sweeper.n_trials={cnn_trials_override}')

# Optional speed/cost controls for sweeps:
# cnn_extra += ['training.total_episodes=12000', 'training.eval_interval=500', 'training.eval_episodes=50']

cnn_code = run_cmd(build_sweep_cmd('sweep_cnn', cnn_extra))
if cnn_code != 0:
    raise RuntimeError('CNN sweep failed. Inspect logs above.')

## 5. Optuna Dashboard

In [ ]:
DASHBOARD_HOST = '127.0.0.1'
DASHBOARD_PORT = 8081
dashboard_url = f'http://{DASHBOARD_HOST}:{DASHBOARD_PORT}'

if 'dashboard_proc' in globals() and dashboard_proc and dashboard_proc.poll() is None:
    print(f'Dashboard already running at {dashboard_url}')
else:
    cmd = [
        sys.executable,
        '-m',
        'optuna_dashboard',
        str(DB_URL),
        '--host',
        DASHBOARD_HOST,
        '--port',
        str(DASHBOARD_PORT),
    ]
    dashboard_proc = subprocess.Popen(cmd, cwd=str(WORKDIR))
    print(f'optuna-dashboard started at {dashboard_url}')
    print('Keep this kernel alive while dashboard is running.')

In [ ]:
if 'dashboard_proc' in globals() and dashboard_proc and dashboard_proc.poll() is None:
    dashboard_proc.send_signal(signal.SIGTERM)
    dashboard_proc.wait(timeout=10)
    print('Dashboard stopped.')
else:
    print('No running dashboard process found in this kernel.')

## 6. Study Summary

Read best trials directly from SQLite.

In [ ]:
def show_study(study_name: str, top_k: int = 5) -> None:
    study = optuna.load_study(study_name=study_name, storage=DB_URL)
    print(f'\nStudy: {study_name}')
    print(f'  Trials: {len(study.trials)}')
    if len(study.trials) == 0:
        print('  No trials yet.')
        return

    print(f'  Best value (avg_return): {study.best_value:.4f}')
    print(f'  Best params: {json.dumps(study.best_params, indent=2)}')

    completed = [t for t in study.trials if t.state.name == 'COMPLETE']
    completed = sorted(completed, key=lambda t: t.value if t.value is not None else float('-inf'), reverse=True)
    print(f'  Top {min(top_k, len(completed))} trials:')
    for t in completed[:top_k]:
        print(f'    Trial {t.number}: value={t.value:.4f}, params={t.params}')

for s in ['samegame_5x5c3s2_mlp', 'samegame_5x5c3s2_cnn']:
    try:
        show_study(s, top_k=5)
    except Exception as exc:
        print(f'\nStudy {s} not available yet: {exc}')

## Notes

- Known caveat: avoid explicit repeated env close/reopen loops with the C backend in ad-hoc code.
- For long CNN sweeps, monitor GPU memory and keep `hydra.sweeper.n_jobs=1`.
- Sweep trials can be resumed by rerunning the same sweep command (plugin uses `load_if_exists=True` internally).